In [1]:
!pip install nltk

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [3]:
document = """About Nepal
What is Nepal?
Nepal is a landlocked country located in South Asia, between India and China. It is known for its rich culture, beautiful mountains, and warm hospitality.

What is the capital of Nepal?
The capital city of Nepal is Kathmandu.

What is the national language of Nepal?
The national language of Nepal is Nepali.

What is the currency of Nepal?
The currency of Nepal is Nepalese Rupee (NPR).

What is Nepal famous for?
Nepal is famous for Mount Everest, Lumbini (the birthplace of Lord Buddha), its diverse culture, and breathtaking Himalayan landscapes.

Where is Mount Everest located?
Mount Everest, the world’s highest peak, is located in the Solukhumbu district of Nepal.

What is the population of Nepal?
According to the 2021 census, Nepal has a population of around 30 million people.

What is the time zone of Nepal?
Nepal Standard Time (NST) is UTC +5:45.

What type of government does Nepal have?
Nepal is a federal democratic republic with a multi-party system.

What is the main religion of Nepal?
Hinduism is the major religion in Nepal, followed by Buddhism, Islam, and Christianity.

What are some popular tourist destinations in Nepal?
Popular destinations include Kathmandu Valley, Pokhara, Chitwan National Park, Lumbini, and the Everest Base Camp trek.

What is the national flower of Nepal?
The national flower of Nepal is the Rhododendron (Lali Gurans).

What is the best time to visit Nepal?
The best time to visit Nepal is during spring (March to May) and autumn (September to November) when the weather is clear and pleasant.

Can foreigners visit Nepal easily?
Yes, foreigners can easily visit Nepal. Tourist visas are available on arrival for most countries.

How can I travel inside Nepal?
You can travel by bus, taxi, motorcycle, or domestic flights between major cities.

What are some traditional Nepali foods?
Popular Nepali foods include Dal Bhat (rice and lentil soup), Momo (dumplings), Thukpa, and Sel Roti.

What festivals are celebrated in Nepal?
Major festivals include Dashain, Tihar, Holi, Buddha Jayanti, and Teej.

Is Nepal safe for tourists
Yes, Nepal is generally safe for tourists. However, travelers should be cautious in remote trekking areas and follow local guidance.

What is the internet speed like in Nepal?
Internet speed varies by area, with fiber connections in cities and 4G networks in most towns. ISPs like WorldLink, Vianet, and ClassicTech provide good coverage.

What is the education system like in Nepal?
Nepal follows a 10+2 education system with primary, secondary, and higher education levels. Universities like Tribhuvan University and Kathmandu University are well known.

Where can I contact Nepal Tourism Board?
You can contact the Nepal Tourism Board via their website - https://www.welcomenepal.com/

Does it snow in Nepal?
Yes, snow is common in the mountainous regions and during winter in areas above 2,500 meters altitude.

What is the national animal of Nepal?
The national animal of Nepal is the cow.

What is the dialing code for Nepal?
The country code for Nepal is +977.

What are some famous mountains of Nepal besides Everest?
Other famous peaks include Kanchenjunga, Lhotse, Makalu, and Annapurna.

What are Nepal’s neighboring countries?
Nepal shares borders with India to the south, east, and west, and China (Tibet) to the north.

What are the main exports of Nepal?
Nepal’s main exports include carpets, handicrafts, tea, pashmina, and herbs.

What are the main means of livelihood in Nepal?
Agriculture is the main source of livelihood for most Nepalese people.

Where was Lord Buddha born?
Lord Buddha was born in Lumbini, which is located in the Rupandehi district of Nepal.

What is the national game of Nepal?
The national game of Nepal is Volleyball.

Where can I learn more about Nepal’s culture and travel?
You can explore more about Nepal’s culture, history, and travel at the official website - https://www.welcomenepal.com/ """


In [4]:
# Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
# tokenize
tokens = word_tokenize(document.lower())

In [6]:
# build vocab
vocab = {'<unk>':0}

for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

vocab

{'<unk>': 0,
 'about': 1,
 'nepal': 2,
 'what': 3,
 'is': 4,
 '?': 5,
 'a': 6,
 'landlocked': 7,
 'country': 8,
 'located': 9,
 'in': 10,
 'south': 11,
 'asia': 12,
 ',': 13,
 'between': 14,
 'india': 15,
 'and': 16,
 'china': 17,
 '.': 18,
 'it': 19,
 'known': 20,
 'for': 21,
 'its': 22,
 'rich': 23,
 'culture': 24,
 'beautiful': 25,
 'mountains': 26,
 'warm': 27,
 'hospitality': 28,
 'the': 29,
 'capital': 30,
 'of': 31,
 'city': 32,
 'kathmandu': 33,
 'national': 34,
 'language': 35,
 'nepali': 36,
 'currency': 37,
 'nepalese': 38,
 'rupee': 39,
 '(': 40,
 'npr': 41,
 ')': 42,
 'famous': 43,
 'mount': 44,
 'everest': 45,
 'lumbini': 46,
 'birthplace': 47,
 'lord': 48,
 'buddha': 49,
 'diverse': 50,
 'breathtaking': 51,
 'himalayan': 52,
 'landscapes': 53,
 'where': 54,
 'world': 55,
 '’': 56,
 's': 57,
 'highest': 58,
 'peak': 59,
 'solukhumbu': 60,
 'district': 61,
 'population': 62,
 'according': 63,
 'to': 64,
 '2021': 65,
 'census': 66,
 'has': 67,
 'around': 68,
 '30': 69,
 'mi

In [7]:
len(vocab)

270

In [8]:
input_sentences = document.split('\n')

In [9]:
def text_to_indices(sentence, vocab):

  numerical_sentence = []

  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])

  return numerical_sentence


In [10]:
input_numerical_sentences = []

for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))


In [11]:
len(input_numerical_sentences)

93

In [12]:
training_sequence = []
for sentence in input_numerical_sentences:

  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])

In [13]:
len(training_sequence)

712

In [14]:
training_sequence[:5]

[[1, 2], [3, 4], [3, 4, 2], [3, 4, 2, 5], [2, 4]]

In [15]:
len_list = []

for sequence in training_sequence:
  len_list.append(len(sequence))

max(len_list)

30

In [16]:
training_sequence[0]

[1, 2]

In [17]:
padded_training_sequence = []
for sequence in training_sequence:

  padded_training_sequence.append([0]*(max(len_list) - len(sequence)) + sequence)

In [18]:
len(padded_training_sequence[10])

30

In [19]:
padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)

In [20]:
padded_training_sequence

tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   0,   3,   4],
        [  0,   0,   0,  ...,   3,   4,   2],
        ...,
        [  0,   0,   0,  ..., 215, 216, 217],
        [  0,   0,   0,  ..., 216, 217, 218],
        [  0,   0,   0,  ..., 217, 218, 219]])

In [21]:
X = padded_training_sequence[:, :-1]
y = padded_training_sequence[:,-1]

In [22]:
X

tensor([[  0,   0,   0,  ...,   0,   0,   1],
        [  0,   0,   0,  ...,   0,   0,   3],
        [  0,   0,   0,  ...,   0,   3,   4],
        ...,
        [  0,   0,   0,  ..., 269, 215, 216],
        [  0,   0,   0,  ..., 215, 216, 217],
        [  0,   0,   0,  ..., 216, 217, 218]])

In [23]:
y

tensor([  2,   4,   2,   5,   4,   6,   7,   8,   9,  10,  11,  12,  13,  14,
         15,  16,  17,  18,  19,   4,  20,  21,  22,  23,  24,  13,  25,  26,
         13,  16,  27,  28,  18,   4,  29,  30,  31,   2,   5,  30,  32,  31,
          2,   4,  33,  18,   4,  29,  34,  35,  31,   2,   5,  34,  35,  31,
          2,   4,  36,  18,   4,  29,  37,  31,   2,   5,  37,  31,   2,   4,
         38,  39,  40,  41,  42,  18,   4,   2,  43,  21,   5,   4,  43,  21,
         44,  45,  13,  46,  40,  29,  47,  31,  48,  49,  42,  13,  22,  50,
         24,  13,  16,  51,  52,  53,  18,   4,  44,  45,   9,   5,  45,  13,
         29,  55,  56,  57,  58,  59,  13,   4,   9,  10,  29,  60,  61,  31,
          2,  18,   4,  29,  62,  31,   2,   5,  64,  29,  65,  66,  13,   2,
         67,   6,  62,  31,  68,  69,  70,  71,  18,   4,  29,  72,  73,  31,
          2,   5,  74,  72,  40,  75,  42,   4,  76,   0,  18,  78,  31,  79,
         80,   2,  81,   5,   4,   6,  82,  83,  84,  85,   6,  

In [24]:
class CustomDataset(Dataset):

  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [25]:
dataset = CustomDataset(X,y)

In [26]:
len(dataset)

712

In [27]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [28]:
class LSTMModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(100, 150, batch_first=True)
    self.fc = nn.Linear(150, vocab_size)

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))
    return output

In [29]:
model = LSTMModel(len(vocab))

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [31]:
model.to(device)

LSTMModel(
  (embedding): Embedding(270, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=270, bias=True)
)

In [32]:
epochs = 50
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [33]:
# training loop

for epoch in range(epochs):
  total_loss = 0

  for batch_x, batch_y in dataloader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output, batch_y)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 124.4816
Epoch: 2, Loss: 106.5594
Epoch: 3, Loss: 95.7992
Epoch: 4, Loss: 87.1710
Epoch: 5, Loss: 80.7527
Epoch: 6, Loss: 74.5791
Epoch: 7, Loss: 68.2513
Epoch: 8, Loss: 61.8785
Epoch: 9, Loss: 56.2982
Epoch: 10, Loss: 51.0950
Epoch: 11, Loss: 45.8298
Epoch: 12, Loss: 40.9068
Epoch: 13, Loss: 37.4744
Epoch: 14, Loss: 33.5479
Epoch: 15, Loss: 29.9675
Epoch: 16, Loss: 26.9025
Epoch: 17, Loss: 24.2361
Epoch: 18, Loss: 21.7579
Epoch: 19, Loss: 19.8244
Epoch: 20, Loss: 18.2112
Epoch: 21, Loss: 16.8570
Epoch: 22, Loss: 14.9141
Epoch: 23, Loss: 14.0142
Epoch: 24, Loss: 13.1087
Epoch: 25, Loss: 12.2712
Epoch: 26, Loss: 11.4812
Epoch: 27, Loss: 10.7914
Epoch: 28, Loss: 10.0327
Epoch: 29, Loss: 9.5573
Epoch: 30, Loss: 9.3845
Epoch: 31, Loss: 8.8782
Epoch: 32, Loss: 8.5980
Epoch: 33, Loss: 7.9038
Epoch: 34, Loss: 8.0312
Epoch: 35, Loss: 7.8780
Epoch: 36, Loss: 7.1684
Epoch: 37, Loss: 7.1364
Epoch: 38, Loss: 7.1761
Epoch: 39, Loss: 6.9367
Epoch: 40, Loss: 6.4232
Epoch: 41, Loss: 6.

In [34]:
# prediction

def prediction(model, vocab, text):

  # tokenize
  tokenized_text = word_tokenize(text.lower())

  # text -> numerical indices
  numerical_text = text_to_indices(tokenized_text, vocab)

  # padding
  padded_text = torch.tensor([0] * (61 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)

  # send to model
  output = model(padded_text)

  # predicted index
  value, index = torch.max(output, dim=1)

  # merge with text
  return text + " " + list(vocab.keys())[index]



In [40]:
prediction(model, vocab, "nepal is a country")

'nepal is a country located'

In [41]:
import time

num_tokens = 10
input_text = "nepal is "

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text
  time.sleep(0.5)


nepal is  a
nepal is  a landlocked
nepal is  a landlocked country
nepal is  a landlocked country located
nepal is  a landlocked country located in
nepal is  a landlocked country located in south
nepal is  a landlocked country located in south asia
nepal is  a landlocked country located in south asia ,
nepal is  a landlocked country located in south asia , between
nepal is  a landlocked country located in south asia , between india


In [37]:
dataloader1 = DataLoader(dataset, batch_size=32, shuffle=False)

In [42]:
# Function to calculate accuracy
def calculate_accuracy(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No need to compute gradients
        for batch_x, batch_y in dataloader1:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            # Get model predictions
            outputs = model(batch_x)

            # Get the predicted word indices
            _, predicted = torch.max(outputs, dim=1)

            # Compare with actual labels
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = correct / total * 100
    return accuracy

# Compute accuracy
accuracy = calculate_accuracy(model, dataloader, device)
print(f"Model Accuracy: {accuracy:.2f}%")


Model Accuracy: 92.98%
